<a href="https://colab.research.google.com/github/VintaBytes/Ciencia-de-datos/blob/main/libros/ciencia-de-datos-con-python/volumen-01/cuadernos/capitulo-013-que-significa-limpiar-datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Capítulo 13 — Qué significa limpiar datos

## Breve repaso

Hasta ahora aprendimos varias operaciones fundamentales para trabajar con `DataFrames` en Pandas. Seleccionamos columnas, seleccionamos filas, usamos `loc` e `iloc`, filtramos registros, combinamos condiciones, ordenamos datos, construimos rankings, creamos nuevas columnas, modificamos valores existentes y organizamos la estructura de una tabla.

Ese recorrido nos dio herramientas importantes. Sin embargo, todavía trabajamos principalmente con datasets pequeños y controlados, creados especialmente para aprender una operación por vez.

A partir de este capítulo vamos a iniciar una nueva etapa: la preparación y limpieza de datos.

En situaciones reales, los datasets rara vez llegan listos para analizar. Pueden tener valores faltantes, columnas con tipos de datos incorrectos, fechas cargadas como texto, números escritos de manera inconsistente, categorías mezcladas, duplicados, errores de carga o valores que no tienen sentido dentro del contexto del problema.

Limpiar datos no significa simplemente “borrar lo que está mal”. Significa revisar el dataset, comprender sus problemas, tomar decisiones justificadas, aplicar transformaciones y verificar que el resultado sea más confiable para el análisis.

En este capítulo vamos a trabajar con una mirada principalmente conceptual y metodológica. Vamos a presentar qué significa limpiar datos, por qué es necesario hacerlo, qué tipos de problemas pueden aparecer y por qué cada decisión de limpieza debe tomarse con cuidado.

Al finalizar este capítulo deberías poder:

- Explicar qué significa limpiar datos.
- Reconocer que la limpieza de datos es una etapa de análisis, no solo de programación.
- Identificar problemas frecuentes en datasets reales.
- Comprender la diferencia entre detectar un problema y decidir cómo tratarlo.
- Reconocer que eliminar datos no siempre es la mejor solución.
- Comprender la importancia de validar el dataset después de limpiarlo.
- Entender el flujo general: diagnosticar, decidir, transformar y verificar.

## Un dataset real para trabajar la limpieza

Para estudiar limpieza de datos vamos a usar un dataset real disponible en Kaggle: **Cafe Sales — Dirty Data for Cleaning Training**.

Este dataset contiene registros de ventas de una cafetería. Cada fila representa una transacción, y las columnas describen aspectos de esa venta, como el producto vendido, la cantidad, el precio unitario, el total gastado, el método de pago, la ubicación y la fecha de la transacción.

A diferencia de los datasets pequeños que usamos hasta ahora, este archivo está pensado para practicar limpieza de datos. Eso significa que no todos sus valores están listos para ser analizados directamente. Algunas columnas pueden tener datos faltantes, valores inconsistentes, tipos de datos incorrectos o registros que requieren revisión.

Esto lo vuelve muy útil para esta etapa del recorrido.

A partir de ahora, el objetivo no será solamente aprender una instrucción nueva de Pandas. Vamos a trabajar sobre un problema más realista: recibimos un dataset imperfecto y necesitamos prepararlo para que pueda ser analizado con mayor confianza.

Esto implica hacernos preguntas como:

```text
¿Qué problemas tiene el dataset?
¿Qué columnas necesitan revisión?
¿Qué valores faltan?
¿Qué valores parecen inconsistentes?
¿Qué tipos de datos fueron interpretados correctamente?
¿Qué columnas se pueden usar para verificar otras?
¿Qué decisiones de limpieza son razonables?
¿Qué riesgos tiene cada decisión?
¿Cómo sabemos si el dataset quedó mejor preparado?
```

La limpieza de datos no es una etapa mecánica. No alcanza con aplicar comandos como `dropna()`, `fillna()` o `replace()` sin pensar. Cada transformación debe tener una justificación y debe ser revisada después de aplicarse.

Por eso, en esta parte vamos a avanzar con más atención conceptual. El código seguirá siendo importante, pero estará al servicio de un proceso: comprender, diagnosticar, limpiar y validar.

## Limpiar datos no es solo aplicar comandos

Cuando empezamos a trabajar con Pandas, puede parecer que limpiar datos consiste simplemente en conocer algunos métodos:

```text
dropna()
fillna()
replace()
astype()
to_datetime()
drop_duplicates()
```

Esos métodos son importantes, y vamos a usarlos más adelante. Pero la limpieza de datos no empieza por el comando. Empieza por una pregunta:

```text
¿Qué problema estoy tratando de resolver?
```

Por ejemplo, si una columna tiene valores faltantes, podríamos pensar rápidamente en eliminarlos o completarlos. Sin embargo, antes de decidir qué hacer necesitamos entender qué representan esos valores faltantes.

No es lo mismo que falte el método de pago en una venta, que falte la cantidad vendida, que falte la fecha de la transacción o que falte el nombre del producto. Cada caso puede tener consecuencias diferentes para el análisis.

También necesitamos preguntarnos por qué falta ese dato. Tal vez hubo un error de carga. Tal vez el dato no correspondía. Tal vez el sistema no lo registró. Tal vez aparece como faltante porque fue escrito de una forma que Pandas no interpretó correctamente.

Por eso, limpiar datos implica tomar decisiones. Algunas decisiones pueden ser simples, como quitar espacios innecesarios en una columna de texto. Otras pueden ser más delicadas, como eliminar filas, completar valores faltantes o modificar tipos de datos.

Un buen proceso de limpieza suele seguir este recorrido:

```text
diagnosticar el problema
        ↓
decidir una estrategia
        ↓
aplicar una transformación
        ↓
verificar el resultado
```

Si omitimos la etapa de diagnóstico, podemos aplicar soluciones incorrectas. Si omitimos la etapa de verificación, podemos creer que el problema fue resuelto cuando en realidad sigue presente o fue reemplazado por otro.

Por eso, a partir de ahora vamos a trabajar con una idea central: cada transformación sobre un dataset debe tener un motivo y debe ser revisada después de aplicarse.

## Problemas frecuentes en datasets reales

Un dataset puede tener muchos tipos de problemas. Algunos son visibles rápidamente, mientras que otros aparecen recién cuando intentamos analizar, filtrar, calcular o graficar.

Uno de los problemas más conocidos son los **valores faltantes**. Un valor faltante aparece cuando una celda no tiene información disponible. En Pandas, muchas veces estos valores se representan como `NaN`. Los valores faltantes pueden afectar conteos, promedios, filtros y gráficos, pero no siempre deben tratarse de la misma manera. A veces conviene eliminarlos, otras veces completarlos, y en algunos casos puede ser mejor conservarlos como evidencia de que la información no está disponible.

También pueden aparecer **duplicados**. Un duplicado ocurre cuando una misma observación aparece más de una vez. En algunos datasets, una fila duplicada puede ser un error de carga. En otros casos, dos filas parecidas pueden representar eventos reales distintos. Por eso, antes de eliminar duplicados, necesitamos entender qué representa cada fila.

Otro problema frecuente son las **inconsistencias en categorías o textos**. Por ejemplo, una misma categoría puede aparecer escrita como `"Credito"`, `"credito"`, `"Crédito"` o `" credito "`. Para una persona, esos valores pueden representar lo mismo. Para Pandas, en cambio, son valores diferentes. Si no corregimos esas diferencias, los conteos y agrupamientos pueden quedar fragmentados.

También pueden aparecer **tipos de datos incorrectos**. Una columna que debería ser numérica puede estar cargada como texto. Una fecha puede aparecer como `object` en lugar de tener un tipo temporal. Una columna con importes puede incluir símbolos, comas, espacios o palabras que impiden convertirla directamente a número.

Además, pueden existir **valores imposibles o sospechosos**. Por ejemplo, una cantidad vendida negativa, un precio igual a cero, una fecha futura en un registro histórico o un total gastado que no coincide con la cantidad multiplicada por el precio unitario. Estos casos no siempre son errores, pero deben llamar nuestra atención.

Finalmente, puede haber **problemas estructurales**, como nombres de columnas poco claros, columnas redundantes, variables mal separadas o información mezclada en una misma columna.

Podemos resumir algunos problemas frecuentes así:

| Problema | Ejemplo | Posible impacto |
|---|---|---|
| Valores faltantes | Falta el método de pago de una venta | Puede afectar conteos o análisis por categoría |
| Duplicados | La misma transacción aparece dos veces | Puede inflar ventas o cantidades |
| Categorías inconsistentes | `"Credito"` y `"credito"` | Fragmenta conteos y agrupamientos |
| Tipos incorrectos | Un precio cargado como texto | Impide cálculos numéricos confiables |
| Fechas mal interpretadas | Una fecha leída como texto | Dificulta ordenar o agrupar temporalmente |
| Valores sospechosos | Cantidad negativa o precio cero | Puede indicar errores de carga |
| Columnas poco claras | Nombres como `col1` o `dato` | Dificulta comprender el dataset |

Detectar estos problemas es solo el primer paso. La parte más importante es decidir qué hacer con ellos de manera razonada.

## Diagnosticar antes de limpiar

Cuando encontramos un problema en un dataset, es tentador corregirlo de inmediato. Si vemos valores faltantes, podemos querer eliminarlos. Si vemos categorías mal escritas, podemos querer reemplazarlas. Si vemos una columna con tipo incorrecto, podemos querer convertirla rápidamente.

Pero antes de modificar el dataset conviene diagnosticar.

Diagnosticar significa observar el problema con más detalle para entender su alcance, su posible causa y su impacto sobre el análisis.

Por ejemplo, si una columna tiene valores faltantes, no alcanza con saber que existen. También necesitamos saber cuántos faltan, qué porcentaje representan, en qué columnas aparecen, si se concentran en cierto grupo, si afectan variables importantes o si pueden reconstruirse a partir de otras columnas.

No es lo mismo que falte el 1% de los valores de una columna secundaria que el 60% de los valores de una columna central para el análisis. Tampoco es lo mismo que los valores faltantes estén distribuidos al azar que concentrados en una sucursal, una fecha o un tipo de producto.

Algo parecido ocurre con los duplicados. Antes de eliminarlos, necesitamos preguntarnos si realmente son duplicados o si representan transacciones distintas que comparten algunos datos. Dos ventas del mismo producto, con el mismo precio y la misma fecha, pueden parecer repetidas, pero tal vez sean ventas reales diferentes.

El diagnóstico nos ayuda a evitar decisiones automáticas.

Una buena pregunta inicial no es:

```text
¿Cómo borro este problema?
```

sino:

```text
¿Qué representa este problema dentro del dataset?
```

A partir de esa respuesta podemos decidir mejor qué hacer. En limpieza de datos, la calidad de la decisión suele ser más importante que la rapidez con la que aplicamos una instrucción.

## Decidir una estrategia de limpieza

Después de diagnosticar un problema, necesitamos decidir qué hacer.

En limpieza de datos, una misma situación puede tener distintas estrategias posibles. Por ejemplo, si encontramos valores faltantes, podríamos eliminar las filas incompletas, eliminar una columna con demasiados faltantes, completar los valores con una medida estadística, reconstruirlos a partir de otras columnas o conservarlos si la ausencia de información también es relevante.

Ninguna de esas opciones es correcta en todos los casos. La decisión depende del contexto, del objetivo del análisis, de la cantidad de datos afectados y de la importancia de la columna.

Por ejemplo, si en un dataset de ventas falta el método de pago en algunas filas, tal vez podamos conservar esas ventas y simplemente registrar que el método de pago es desconocido. Pero si falta la cantidad vendida o el precio unitario, el problema puede ser más serio, porque esas columnas son necesarias para calcular importes.

También puede ocurrir que un valor faltante pueda reconstruirse. Si tenemos `Quantity`, `Price Per Unit` y `Total Spent`, y falta solo uno de esos tres valores, quizá podamos deducirlo usando los otros dos. En ese caso, completar el dato no sería una simple imputación arbitraria, sino una reconstrucción basada en una relación interna del dataset.

Con los duplicados ocurre algo parecido. Si dos filas son exactamente iguales, tal vez una de ellas sea una copia accidental. Pero si dos filas tienen el mismo producto, el mismo precio y la misma fecha, eso no alcanza para decir que son duplicadas: podrían representar dos ventas reales distintas.

Por eso, antes de limpiar conviene formular una estrategia. Esa estrategia debería responder preguntas como:

```text
¿Qué problema detectamos?
¿Qué columnas están afectadas?
¿Cuántas filas están involucradas?
¿Qué impacto tiene el problema sobre el análisis?
¿Qué opciones de tratamiento tenemos?
¿Qué opción conserva mejor la información?
¿Qué riesgos tiene la decisión elegida?
¿Cómo vamos a verificar el resultado?
```

La limpieza de datos no debería ser un conjunto de acciones automáticas. Es un proceso de toma de decisiones documentadas.

## Verificar después de transformar

Una limpieza no termina cuando aplicamos una transformación. Termina cuando verificamos que esa transformación produjo el resultado esperado.

Este punto es fundamental. Podemos escribir una instrucción correcta desde el punto de vista sintáctico y, aun así, obtener un resultado que no resuelve el problema. También puede ocurrir que corrijamos una columna, pero sin darnos cuenta generemos un problema nuevo en otra parte del dataset.

Por ejemplo, si unificamos categorías, deberíamos revisar nuevamente los valores únicos o los conteos para confirmar que las variantes quedaron agrupadas. Si convertimos una columna a tipo numérico, deberíamos revisar cuántos valores no pudieron convertirse. Si eliminamos duplicados, deberíamos comprobar cuántas filas quedaron antes y después. Si imputamos valores faltantes, deberíamos verificar que la cantidad de faltantes disminuyó y que la imputación no generó valores incoherentes.

La verificación posterior responde una pregunta muy simple:

```text
¿Cómo sé que la limpieza funcionó?
```

En Pandas, algunas herramientas útiles para verificar transformaciones son:

| Herramienta          | Para qué puede servir                                 |
| -------------------- | ----------------------------------------------------- |
| `head()`             | Observar las primeras filas después de un cambio      |
| `info()`             | Revisar tipos de datos y valores no nulos             |
| `isna().sum()`       | Contar valores faltantes por columna                  |
| `value_counts()`     | Revisar categorías y frecuencias                      |
| `unique()`           | Observar valores distintos en una columna             |
| `duplicated().sum()` | Contar filas duplicadas                               |
| `describe()`         | Revisar estadísticas de columnas numéricas            |
| `shape`              | Comparar cantidad de filas y columnas antes y después |

La validación posterior es una forma de control de calidad. Nos permite comprobar que el dataset quedó más preparado para el análisis, y no simplemente diferente.

Por eso, una buena limpieza debería dejar algún tipo de evidencia: una comparación, una tabla de revisión, un conteo antes y después, o una comprobación clara de que el problema fue tratado correctamente.

## Un flujo general para limpiar datos

Aunque cada dataset tiene sus propios problemas, podemos pensar la limpieza de datos como un proceso ordenado.

No siempre vamos a seguir exactamente los mismos pasos ni en el mismo orden, pero sí conviene tener una guía general. Esa guía nos ayuda a no limpiar datos de manera improvisada.

Una posible secuencia de trabajo es:

```text
cargar el dataset
        ↓
inspeccionar su estructura
        ↓
detectar problemas
        ↓
priorizar qué problemas tratar
        ↓
aplicar transformaciones
        ↓
verificar los cambios
        ↓
documentar decisiones
        ↓
obtener una versión preparada para analizar
```

En la práctica, este recorrido no siempre es lineal. Muchas veces detectamos un problema, lo corregimos, verificamos el resultado y eso nos permite encontrar otro problema que antes no era visible. La limpieza puede requerir varias idas y vueltas.

Por ejemplo, al convertir una columna a tipo numérico podemos descubrir que algunos valores contenían texto. Al limpiar una columna categórica podemos encontrar categorías que no esperábamos. Al revisar duplicados podemos descubrir que necesitamos mirar más de una columna para decidir si dos registros son realmente iguales.

Por eso, más que una receta cerrada, este flujo debe entenderse como una forma de trabajo:

```text
observar → decidir → transformar → verificar
```

A medida que avancemos, vamos a aplicar esta lógica sobre un dataset real. No vamos a limpiar todo de una vez. Vamos a separar los problemas, analizarlos y tomar decisiones paso a paso.

## Cargar el dataset real

Para comenzar a trabajar con un dataset real, vamos a descargarlo desde Kaggle usando la biblioteca `kagglehub`.

Este paso puede tardar unos segundos la primera vez que se ejecuta. Una vez descargado el dataset, vamos a localizar el archivo CSV y cargarlo con Pandas.

En este capítulo no vamos a resolver todos los problemas del dataset. La idea es observarlo por primera vez y comenzar a pensar qué tipo de limpieza podría necesitar.

In [ ]:
# Instalamos kagglehub si no está disponible en el entorno.
# En Google Colab, esta instalación suele ser necesaria.

!pip install kagglehub

In [ ]:
import kagglehub
import pandas as pd
import os

# Descargamos el dataset desde Kaggle.
# KaggleHub devuelve la ruta local donde quedó almacenado.

ruta_dataset = kagglehub.dataset_download(
    "ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training"
)

ruta_dataset

In [ ]:
# Revisamos qué archivos se descargaron dentro de la carpeta del dataset.

archivos = os.listdir(ruta_dataset)

archivos

['dirty_cafe_sales.csv']

La celda anterior muestra los archivos disponibles dentro de la carpeta descargada.

Como el dataset puede venir acompañado por más de un archivo, primero revisamos el contenido de la carpeta y luego elegimos el archivo CSV que vamos a cargar.

In [ ]:
# Buscamos archivos CSV dentro de la carpeta descargada.

archivos_csv = [archivo for archivo in archivos if archivo.endswith(".csv")]

archivos_csv

['dirty_cafe_sales.csv']

In [ ]:
# Cargamos el primer archivo CSV encontrado.

ruta_csv = os.path.join(ruta_dataset, archivos_csv[0])

df = pd.read_csv(ruta_csv)

df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


Ahora tenemos el dataset cargado en un `DataFrame` llamado `df`.

La salida de `head()` nos muestra las primeras filas. Esta primera vista no alcanza para diagnosticar todos los problemas, pero nos permite confirmar que el archivo fue cargado correctamente y empezar a reconocer sus columnas.

A partir de este punto, el trabajo de limpieza comienza con una pregunta simple: ¿qué tenemos delante?

## Primera inspección del dataset

Una vez cargado el archivo, necesitamos hacer una primera inspección general.

En esta etapa no vamos a limpiar todavía. Solo queremos observar la estructura del dataset: cuántas filas y columnas tiene, cómo se llaman las columnas, qué tipos de datos interpretó Pandas y qué primeras señales de problemas aparecen.

Vamos a comenzar revisando las dimensiones del `DataFrame`.

In [ ]:
df.shape

(10000, 8)

La salida de `shape` nos indica cuántas filas y cuántas columnas tiene el dataset.

Como estamos trabajando con un archivo real, ya no se trata de una tabla pequeña creada manualmente. Este tamaño nos obliga a trabajar con herramientas de inspección y resumen, porque no podemos revisar cada fila una por una.

In [ ]:
df.columns

Index(['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent',
       'Payment Method', 'Location', 'Transaction Date'],
      dtype='object')

La salida de `columns` muestra los nombres de las columnas disponibles.

A partir de esos nombres podemos empezar a comprender qué información contiene el dataset. En este caso, las columnas describen transacciones de venta: identificador de transacción, producto, cantidad, precio unitario, total gastado, método de pago, ubicación y fecha.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


La salida de `info()` nos da una primera imagen de la estructura del dataset.

Nos permite observar cuántos valores no nulos tiene cada columna y qué tipo de dato interpretó Pandas. Esta información será importante para detectar posibles problemas, como columnas con valores faltantes o columnas numéricas que fueron cargadas como texto.

En esta primera lectura todavía no tomamos decisiones. Solo estamos reuniendo evidencia para entender qué tipo de limpieza puede necesitar el dataset.

## Primeras señales de problemas

Después de revisar la estructura general del dataset, podemos empezar a buscar señales de posibles problemas.

En esta etapa todavía no vamos a resolverlos. Solo queremos detectarlos y registrar qué aspectos conviene analizar con más detalle.

Algunas preguntas iniciales pueden ser:

```text
¿Hay valores faltantes?
¿Hay columnas que parecen numéricas pero fueron cargadas como texto?
¿Hay fechas interpretadas correctamente?
¿Aparecen valores extraños en columnas categóricas?
¿Hay columnas que se puedan usar para verificar otras?
```

Para empezar, podemos observar algunas filas del dataset.


In [ ]:
df.head(10)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,NaN,5,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,NaN,In-store,2023-12-31


Esta vista inicial permite reconocer la forma general de los datos.

Sin embargo, mirar unas pocas filas no alcanza para diagnosticar un dataset completo. Algunos problemas pueden aparecer recién más adelante, o afectar solo a una parte pequeña de las filas.

Por eso, además de observar ejemplos, necesitamos usar herramientas de resumen.

In [ ]:
df.describe(include="all")

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_9226047,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


El método `describe(include="all")` nos da un resumen amplio del dataset, incluyendo columnas numéricas y no numéricas.

En este punto, no necesitamos interpretar cada valor en detalle. Lo importante es empezar a detectar señales: columnas con muchos valores distintos, valores frecuentes, posibles datos faltantes, tipos de datos inesperados o columnas que merecen una revisión más cuidadosa.

La limpieza de datos suele comenzar así: no con una transformación inmediata, sino con una lectura atenta de las señales que el dataset nos ofrece.

## Detectar valores faltantes como diagnóstico inicial

Uno de los primeros problemas que conviene revisar en un dataset real es la presencia de valores faltantes.

Un valor faltante indica que una celda no tiene información disponible. En Pandas, estos valores suelen aparecer como `NaN`.

En este capítulo no vamos a decidir todavía cómo tratar esos valores. Por ahora, solo queremos detectar en qué columnas aparecen y tener una primera idea de su magnitud.

Para eso podemos usar:

In [ ]:
df.isna().sum()

,0
Transaction ID,0
Item,333
Quantity,138
Price Per Unit,179
Total Spent,173
Payment Method,2579
Location,3265
Transaction Date,159


La instrucción `isna()` identifica los valores faltantes del `DataFrame`, y `sum()` permite contar cuántos hay en cada columna.

La salida nos muestra, columna por columna, cuántos valores faltantes fueron detectados por Pandas.

Este conteo es una primera señal importante. No alcanza para decidir qué hacer, pero nos ayuda a saber qué columnas necesitan revisión.

También podemos calcular el porcentaje de valores faltantes por columna:

In [ ]:
porcentaje_faltantes = df.isna().mean() * 100

porcentaje_faltantes

,0
Transaction ID,0.00
Item,3.33
Quantity,1.38
Price Per Unit,1.79
Total Spent,1.73
Payment Method,25.79
Location,32.65
Transaction Date,1.59


Este porcentaje ayuda a interpretar mejor el problema.

No es lo mismo que una columna tenga 5 valores faltantes en un dataset de 10.000 filas que en un dataset de 20 filas. Por eso, además del conteo absoluto, suele ser útil observar la proporción de datos faltantes.

En los próximos pasos vamos a estudiar los valores faltantes con más profundidad. Por ahora, lo importante es registrar que forman parte del diagnóstico inicial del dataset.

## Valores que parecen faltantes pero no siempre son NaN

Cuando usamos `isna()`, Pandas detecta valores faltantes reales, como `NaN`.

Sin embargo, en muchos datasets aparecen valores que representan ausencia de información, pero que fueron escritos como texto. Por ejemplo:

```text
UNKNOWN
ERROR
N/A
Sin dato
No informado
```

Para Pandas, esos valores no siempre son faltantes. Si aparecen como texto dentro de una columna, Pandas puede tratarlos como valores comunes.

Por eso, además de contar `NaN`, conviene revisar los valores únicos de algunas columnas categóricas.

In [ ]:
df["Item"].value_counts(dropna=False)

,count
Item,
Juice,1171
Coffee,1165
Salad,1148
Cake,1139
Sandwich,1131
Smoothie,1096
Cookie,1092
Tea,1089
UNKNOWN,344


In [ ]:
df["Payment Method"].value_counts(dropna=False)

,count
Payment Method,
NaN,2579
Digital Wallet,2291
Credit Card,2273
Cash,2258
ERROR,306
UNKNOWN,293


In [ ]:
df["Location"].value_counts(dropna=False)

,count
Location,
NaN,3265
Takeaway,3022
In-store,3017
ERROR,358
UNKNOWN,338


Usamos `value_counts(dropna=False)` para contar todos los valores, incluyendo posibles valores faltantes.

Esta revisión puede mostrar valores como `"UNKNOWN"` o `"ERROR"`, que no necesariamente aparecen en el conteo de `isna()`, pero que también pueden indicar problemas de calidad de datos.

Esto nos enseña una idea importante: detectar valores faltantes no siempre alcanza con buscar `NaN`. A veces necesitamos conocer cómo fue cargado el dataset y revisar qué valores especiales se usaron para representar datos desconocidos, errores o información no disponible.

En capítulos posteriores vamos a decidir cómo tratar estos casos. Por ahora, los registramos como parte del diagnóstico inicial.

## Tipos de datos que necesitan revisión

Además de los valores faltantes, otro aspecto importante del diagnóstico inicial es revisar los tipos de datos.

Pandas asigna un tipo de dato a cada columna cuando carga el archivo. Sin embargo, esa interpretación no siempre coincide con lo que esperamos.

Por ejemplo, en un dataset de ventas, columnas como `Quantity`, `Price Per Unit` o `Total Spent` deberían poder analizarse como valores numéricos. La columna `Transaction Date`, en cambio, debería poder interpretarse como una fecha.

Podemos revisar los tipos de datos con:

In [ ]:
df.dtypes

,0
Transaction ID,object
Item,object
Quantity,object
Price Per Unit,object
Total Spent,object
Payment Method,object
Location,object
Transaction Date,object


Si una columna que debería ser numérica aparece como `object`, eso puede indicar que contiene textos, símbolos, valores especiales o errores de carga.

Algo parecido ocurre con las fechas. Una columna de fecha puede aparecer inicialmente como `object` porque Pandas la cargó como texto. Eso no significa necesariamente que los datos estén mal, pero sí indica que necesitaremos convertirlos antes de hacer análisis temporal.

En este punto no vamos a convertir todavía las columnas. Solo vamos a registrar qué columnas necesitan revisión.

Podemos observar algunas columnas específicas:

In [ ]:
df[["Quantity", "Price Per Unit", "Total Spent", "Transaction Date"]].head()

,Quantity,Price Per Unit,Total Spent,Transaction Date
0,2,2.0,4.0,2023-09-08
1,4,3.0,12.0,2023-05-16
2,4,1.0,ERROR,2023-07-19
3,2,5.0,10.0,2023-04-27
4,2,2.0,4.0,2023-06-11


Esta vista nos permite mirar columnas que probablemente requieran tratamiento posterior.

Si aparecen valores como textos, errores, símbolos o datos faltantes, eso puede explicar por qué Pandas no interpretó correctamente el tipo de dato.

Más adelante vamos a trabajar con conversiones usando herramientas como `pd.to_numeric()` y `pd.to_datetime()`. Por ahora, nos alcanza con reconocer que revisar los tipos de datos forma parte del diagnóstico inicial.

## Columnas que pueden verificarse entre sí

En algunos datasets, ciertas columnas están relacionadas entre sí.

En este caso, tenemos columnas que deberían guardar una relación matemática:

```text
Total Spent = Quantity × Price Per Unit
```

Esto significa que, si conocemos la cantidad vendida y el precio unitario, podemos calcular el total gastado de una transacción.

Esa relación interna puede ser muy útil durante la limpieza. Por ejemplo, si falta `Total Spent`, pero tenemos `Quantity` y `Price Per Unit`, tal vez podamos reconstruir el valor faltante. Si `Total Spent` está cargado, pero no coincide con la multiplicación entre cantidad y precio unitario, podríamos haber encontrado una inconsistencia.

Todavía no vamos a corregir esos problemas. En esta etapa solo queremos reconocer que el dataset contiene columnas que pueden ayudarnos a validar otras columnas.

Podemos observar algunas filas con esas variables:

In [ ]:
df[["Quantity", "Price Per Unit", "Total Spent"]].head(10)

,Quantity,Price Per Unit,Total Spent
0,2,2.0,4.0
1,4,3.0,12.0
2,4,1.0,ERROR
3,2,5.0,10.0
4,2,2.0,4.0
5,5,4.0,20.0
6,3,3.0,9.0
7,4,4.0,16.0
8,5,3.0,15.0
9,5,4.0,20.0


Esta vista permite empezar a pensar en una validación posterior.

En limpieza de datos, no siempre dependemos de reglas externas. A veces el propio dataset contiene relaciones que nos ayudan a detectar errores, reconstruir valores o comprobar si una transformación fue razonable.

Este punto será muy importante más adelante, cuando trabajemos con valores faltantes, conversiones numéricas y validación posterior a la limpieza.

## Registrar los problemas detectados

Después de una primera inspección, conviene registrar qué problemas o señales encontramos.

Este registro no tiene que ser definitivo. Es una primera lista de aspectos que vamos a revisar con más detalle más adelante.

A partir de las inspecciones realizadas, podríamos anotar problemas como estos:

| Aspecto observado | Posible problema | Qué deberíamos revisar después |
|---|---|---|
| Valores faltantes detectados con `isna()` | Hay celdas sin información disponible | Cantidad, porcentaje e importancia de los faltantes |
| Valores como `UNKNOWN` o `ERROR` | Pueden representar datos desconocidos o errores de carga | En qué columnas aparecen y cómo tratarlos |
| Columnas numéricas cargadas como `object` | Puede haber textos o valores no numéricos mezclados | Conversión con control de errores |
| Fechas cargadas como texto | No se pueden analizar temporalmente de forma adecuada | Conversión a tipo fecha |
| Relación entre `Quantity`, `Price Per Unit` y `Total Spent` | Puede servir para detectar o reconstruir valores | Validar si el total coincide con cantidad por precio |

Este tipo de registro es importante porque evita que la limpieza se convierta en una serie de acciones improvisadas.

Antes de transformar el dataset, necesitamos tener una idea de qué problemas existen y en qué orden conviene tratarlos.

En los próximos capítulos vamos a tomar estos problemas uno por uno. Primero vamos a profundizar en los valores faltantes, luego veremos estrategias para tratarlos, y más adelante trabajaremos con duplicados, textos inconsistentes, tipos de datos, fechas y validación posterior.

## Resumen del capítulo

En este capítulo iniciamos una nueva etapa del recorrido: la preparación y limpieza de datos.

Hasta ahora habíamos trabajado principalmente con operaciones puntuales sobre `DataFrames`: seleccionar, filtrar, ordenar, crear columnas, modificar valores y organizar la estructura de una tabla. En este capítulo cambiamos el enfoque. Empezamos a pensar la limpieza de datos como un proceso completo, no como una lista de comandos aislados.

La idea principal fue que limpiar datos no significa simplemente borrar lo que está mal. Limpiar datos implica diagnosticar problemas, comprender su posible causa, decidir una estrategia, aplicar transformaciones y verificar los resultados.

También vimos que los datasets reales pueden presentar muchos tipos de problemas: valores faltantes, duplicados, categorías inconsistentes, tipos de datos incorrectos, fechas mal interpretadas, valores sospechosos o columnas poco claras.

Un punto importante fue distinguir entre detectar un problema y decidir cómo tratarlo. Saber que una columna tiene valores faltantes, por ejemplo, no nos dice automáticamente si debemos eliminarlos, imputarlos, reconstruirlos o conservarlos. Esa decisión depende del contexto, del objetivo del análisis y del impacto que tenga el problema sobre los resultados.

También presentamos un flujo general de limpieza:

```text
cargar el dataset
        ↓
inspeccionar su estructura
        ↓
detectar problemas
        ↓
priorizar qué problemas tratar
        ↓
aplicar transformaciones
        ↓
verificar los cambios
        ↓
documentar decisiones
        ↓
obtener una versión preparada para analizar
```

Luego cargamos un dataset real desde Kaggle: **Cafe Sales — Dirty Data for Cleaning Training**. A partir de ese dataset hicimos una primera inspección general usando herramientas ya conocidas como `shape`, `columns`, `info()`, `head()` y `describe(include="all")`.

También hicimos un diagnóstico inicial. Revisamos valores faltantes con `isna().sum()`, calculamos porcentajes de faltantes, observamos valores categóricos con `value_counts(dropna=False)`, revisamos tipos de datos con `dtypes` y reconocimos que algunas columnas pueden servir para validar otras. En particular, vimos que en este dataset existe una relación esperada entre `Quantity`, `Price Per Unit` y `Total Spent`.

La idea principal de este capítulo fue:

```text
Limpiar datos es un proceso de diagnóstico, decisión, transformación y validación.
```

A partir de ahora vamos a trabajar sobre problemas concretos del dataset, pero manteniendo esta lógica: antes de modificar los datos, debemos comprender qué problema estamos tratando de resolver.

## Próximo paso

El siguiente paso será estudiar en profundidad los valores faltantes.

Vamos a analizar qué significa que un dato esté ausente, cómo representa Pandas esos valores, cómo detectarlos, cómo contarlos, cómo calcular su proporción y por qué no todos los valores faltantes deben tratarse de la misma manera.

También vamos a empezar a distinguir entre distintos tipos de ausencia: datos realmente faltantes, valores desconocidos escritos como texto y errores de carga que necesitan ser interpretados antes de decidir una estrategia de limpieza.